In [ ]:
## Cell 1 · Imports

import pandas as pd
import numpy as np
import requests
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.neighbors import BallTree

df = pd.read_csv("csv/00_base_data.csv")
print(f"Loaded {len(df)} records")

CENTROID_CACHE = "cache/tract_centroids.json"

In [5]:
## Cell 2 · Get Tract Centroids from Census API (one call)

import requests
import numpy as np
from sklearn.neighbors import BallTree

# Get all NYC census tracts with coordinates in one API call
url = "https://api.census.gov/data/2022/acs/acs5"
params = {
    "get": "B01003_001E,NAME",
    "for": "tract:*",
    "in": "state:36 county:061",  # Manhattan
}
r = requests.get(url, params=params, timeout=60)
data = r.json()

tracts = pd.DataFrame(data[1:], columns=data[0])
tracts["population"] = pd.to_numeric(tracts["B01003_001E"], errors="coerce")
tracts = tracts[["NAME", "state", "county", "tract", "population"]].dropna()
print(f"✓ {len(tracts)} Manhattan tracts loaded")
print(tracts.head(3))

✓ 310 Manhattan tracts loaded
                                           NAME state county   tract  \
0     Census Tract 1; New York County; New York    36    061  000100   
1  Census Tract 2.01; New York County; New York    36    061  000201   
2  Census Tract 2.02; New York County; New York    36    061  000202   

   population  
0           0  
1        2666  
2        7001  


In [ ]:
## Cell 3 · Get Tract Centroids from Census Tiger API (cached + parallel)

def get_tract_centroid(state, county, tract):
    url = "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/Tracts_Blocks/MapServer/0/query"
    params = {
        "where": f"STATE='{state}' AND COUNTY='{county}' AND TRACT='{tract}'",
        "outFields": "TRACT",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "json"
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        data = r.json()
        if data.get("features"):
            rings = data["features"][0]["geometry"]["rings"][0]
            lats = [p[1] for p in rings]
            lons = [p[0] for p in rings]
            return np.mean(lats), np.mean(lons)
    except Exception:
        pass
    return None, None

# ── Load from cache or fetch ─────────────────────────
if os.path.exists(CENTROID_CACHE):
    with open(CENTROID_CACHE, "r") as f:
        cached = json.load(f)
    print(f"Loaded {len(cached)} cached centroids from {CENTROID_CACHE}")
else:
    cached = {}

tracts["_key"] = tracts["state"] + "-" + tracts["county"] + "-" + tracts["tract"]
to_fetch = tracts[~tracts["_key"].isin(cached)].copy()

if len(to_fetch) > 0:
    print(f"Fetching {len(to_fetch)} centroids ({len(cached)} already cached)...")

    def _fetch_one(row):
        lat, lon = get_tract_centroid(row["state"], row["county"], row["tract"])
        return row["_key"], lat, lon

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(_fetch_one, row): row["_key"]
                   for _, row in to_fetch.iterrows()}
        done = 0
        for future in as_completed(futures):
            key, lat, lon = future.result()
            if lat is not None:
                cached[key] = [lat, lon]
            done += 1
            if done % 50 == 0:
                print(f"  {done}/{len(to_fetch)} fetched")

    os.makedirs(os.path.dirname(CENTROID_CACHE), exist_ok=True)
    with open(CENTROID_CACHE, "w") as f:
        json.dump(cached, f)
    print(f"  Saved {len(cached)} centroids to {CENTROID_CACHE}")
else:
    print("All centroids served from cache — no API calls needed")

tracts["lat_t"] = tracts["_key"].map(lambda k: cached[k][0] if k in cached else None)
tracts["lon_t"] = tracts["_key"].map(lambda k: cached[k][1] if k in cached else None)
tracts = tracts.dropna(subset=["lat_t", "lon_t"])
print(f"✓ {len(tracts)} tracts with coordinates")

In [9]:
## Cell 4 · Match Shops to Nearest Tract

tracts["lat_t"] = tracts["lat_t"].astype(float)
tracts["lon_t"] = tracts["lon_t"].astype(float)

tract_coords = np.radians(tracts[["lat_t", "lon_t"]].values)
tree = BallTree(tract_coords, metric="haversine")

shop_coords = np.radians(df[["lat", "lon"]].values)
distances, indices = tree.query(shop_coords, k=1)

df["population_density"] = tracts.iloc[indices.flatten()]["population"].values

print(f"✓ Done")
print(f"  Fill : {df['population_density'].notna().sum()}/{len(df)}")
print(f"  Mean : {df['population_density'].mean():.0f}")
print(f"  Min  : {df['population_density'].min():.0f}")
print(f"  Max  : {df['population_density'].max():.0f}")

✓ Done
  Fill : 2915/2915
  Mean : 2561
  Min  : 89
  Max  : 9612


In [10]:
## Cell 4 · Save

df_out = df[["osm_id", "population_density"]]
df_out.to_csv("csv/12_population_density.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/12_population_density.csv")
print(df_out.describe().round(0))

Saved 2915 records to csv/12_population_density.csv
             osm_id  population_density
count  2.915000e+03              2915.0
mean   6.217806e+09              2561.0
std    4.166958e+09              2443.0
min    8.539400e+07                89.0
25%    2.709934e+09               310.0
50%    5.726058e+09              1636.0
75%    1.023030e+10              3551.0
max    1.380607e+10              9612.0
